# Dark stack

Takes the given nights, finds the darks, checks their headers and keeps one exposure time: the one with the most darks, among those of at least
`MIN_EXPTIME` s with at least `MIN_NDARK` darks. From
every dark the bias is subtracted (the `mean` map of our bias stack of the same nights, made by `bias/bias_stack.ipynb`)
and the result is divided by the exposure time from the dark's own header, so all maps are in **ADU/s**. Then, as for the zeros, the image is cut into
bands and for every pixel the mean, median, std and std_mad of the darks are computed (mean and std with sigma
clipping), plus `std_excess`: the part of the pixel scatter that the bias noise does not explain.

Darks are read from the OCM archive `/data/fits/<tel>/raw/<night>/` and not copied; the maps are saved as FITS files in
`~/data/ocm_dark/<tel>/`.

In [4]:
import os
import glob
import numpy as np
from astropy.io import fits
from astropy.stats import sigma_clip
from tqdm.notebook import tqdm

TEL = 'wk06'
N_START = 1330    # first (oldest) night, int(JD) - 2460000
N_STOP = 1340     # last (newest) night
MIN_EXPTIME = 600  # [s] shortest exposure time of the darks to stack
MIN_NDARK = 4      # fewest darks of one exposure time to stack

RAW_DIR = f'/data/fits/{TEL}/raw'
BIAS_DIR = os.path.expanduser(f'~/data/ocm_bias/{TEL}')
OUT_DIR = os.path.expanduser(f'~/data/ocm_dark/{TEL}')

## Find the darks

In [5]:
f_list = []
exptimes = []
tcams = []
for night in tqdm(range(N_START, N_STOP + 1), desc='nights'):
    files = sorted(glob.glob(f'{RAW_DIR}/{night}/*.fits'))
    for f in tqdm(files, desc=f'night {night}', leave=False):
        h = fits.getheader(f)
        if h['IMAGETYP'] == 'dark':
            f_list.append(f)
            exptimes.append(h['EXPTIME'])
            tcams.append(h['T-CAM'])

print(len(f_list), 'darks found')

nights:   0%|          | 0/11 [00:00<?, ?it/s]

night 1330:   0%|          | 0/338 [00:00<?, ?it/s]

night 1331:   0%|          | 0/11 [00:00<?, ?it/s]

night 1332:   0%|          | 0/81 [00:00<?, ?it/s]

night 1333: 0it [00:00, ?it/s]

night 1334:   0%|          | 0/21 [00:00<?, ?it/s]

... (3 outputs left out)

night 1338:   0%|          | 0/124 [00:00<?, ?it/s]

night 1339: 0it [00:00, ?it/s]

night 1340:   0%|          | 0/190 [00:00<?, ?it/s]

EXPTIME    900 s: 12 darks
EXPTIME   1800 s: 6 darks


## Check the headers

Only darks with the CCD at its set temperature (-60 C). Darks of different exposure times are not mixed (the read
noise in ADU/s differs), so one exposure time is chosen: of those at least `MIN_EXPTIME` s long with at least
`MIN_NDARK` darks, the one with the most darks (with equal numbers the longer one).

In [ ]:
good = []  # (file, exposure time) of the darks with the CCD at its set temperature
for f, t, tcam in zip(f_list, exptimes, tcams):
    if tcam < -59:
        good.append((f, t))
    else:
        print('rejected (T-CAM)', f, tcam)

# exposure times that qualify, as (number of darks, exposure time)
candidates = []
for t in sorted(set(round(t) for f, t in good)):
    n = sum(1 for f, e in good if round(e) == t)
    print(f'EXPTIME {t:6d} s: {n} darks')
    if t >= MIN_EXPTIME and n >= MIN_NDARK:
        candidates.append((n, t))
if not candidates:
    raise ValueError(f'no exposure time with at least {MIN_NDARK} darks of at least {MIN_EXPTIME} s')

n, EXPTIME = max(candidates)  # the most darks; with equal numbers the longer exposure
dark_list = [f for f, t in good if round(t) == EXPTIME]
dark_exptimes = [t for f, t in good if round(t) == EXPTIME]  # EXPTIME from the header of every dark used
print(f'used: EXPTIME {EXPTIME} s, {len(dark_list)} darks')
print(dark_list)

## Bias

The bias stack of the same nights: its `mean` map is subtracted from every dark, its `std` map (the read noise of every
pixel) is used for `std_excess`.

In [ ]:
bias_name = f'{TEL}_{N_START}_{N_STOP}'
bias_mean = fits.getdata(f'{BIAS_DIR}/{bias_name}_mean.fits')
bias_std = fits.getdata(f'{BIAS_DIR}/{bias_name}_std.fits')
print(bias_name, 'level', np.median(bias_mean), 'noise', np.median(bias_std))

## Stack

The same bands as in `bias_stack`. Every band of every dark: (raw - bias mean) / its EXPTIME from the header, in ADU/s.

`std_excess`: the scatter of a pixel over the darks has two parts, the read noise (the same as in the zeros,
`bias_std`) and the noise of the dark signal itself. In variances they add up, so the dark part is
`sqrt(std_dark**2 - bias_std**2)`, both in ADU per exposure, then divided by EXPTIME. Where the difference is negative
(the pixel scatters less than the read noise, i.e. no measurable dark noise) it is set to 0.

In [ ]:
SEG = 256  # height of one band in rows
SIGMA = 5  # mean and std without values further than SIGMA std_mad from the pixel median (cosmics)

ny, nx = bias_mean.shape

mean = np.zeros((ny, nx), np.float32)
median = np.zeros((ny, nx), np.float32)
std = np.zeros((ny, nx), np.float32)
std_mad = np.zeros((ny, nx), np.float32)

for y0 in tqdm(range(0, ny, SEG), desc='bands'):
    y1 = min(y0 + SEG, ny)

    # rows y0..y1 of every dark, bias subtracted, in ADU/s
    paski = []
    for f, t in tqdm(zip(dark_list, dark_exptimes), total=len(dark_list), desc=f'rows {y0}', leave=False):
        with fits.open(f) as hdul:
            paski.append((hdul[0].section[y0:y1, :].astype(np.float32) - bias_mean[y0:y1]) / t)
    paski = np.array(paski)

    # the same statistics as for the zeros (see bias_stack)
    clipped = sigma_clip(paski, sigma=SIGMA, axis=0, stdfunc='mad_std')
    mean[y0:y1] = clipped.mean(axis=0)
    std[y0:y1] = clipped.std(axis=0)
    median[y0:y1] = np.median(paski, axis=0)
    std_mad[y0:y1] = 1.4826 * np.median(np.abs(paski - median[y0:y1]), axis=0)

# dark noise without the read noise; std * exptime = std in ADU per exposure, like bias_std
t = np.mean(dark_exptimes)  # all darks have EXPTIME (selected above), so this is just their exposure time
excess = (std * t) ** 2 - bias_std ** 2
std_excess = (np.sqrt(np.clip(excess, 0, None)) / t).astype(np.float32)

for name, m in [('mean', mean), ('median', median), ('std', std), ('std_mad', std_mad), ('std_excess', std_excess)]:
    print(f'{name:10s} median {np.median(m):10.6f}   min {m.min():10.6f}   max {m.max():10.6f}  ADU/s')

## Save

In [ ]:
# common header: what the maps were made from
hdr = fits.Header()
hdr['TELESCOP'] = TEL
hdr['NIGHT-BG'] = (N_START, 'First night')
hdr['NIGHT-EN'] = (N_STOP, 'Last night')
hdr['EXPTIME'] = (np.mean(dark_exptimes), '[s] Exposure time of the darks, from headers')
hdr['BUNIT'] = 'ADU/s'
hdr['NDARK'] = (len(dark_list), 'Number of darks in the stack')
hdr['BIAS'] = (bias_name, 'Bias stack subtracted (mean map)')
hdr['SIGCLIP'] = (SIGMA, 'Sigma clipping (MAD sigma) of mean and std')
for i, f in enumerate(dark_list):
    hdr[f'FILE{i + 1:03d}'] = os.path.basename(f)

# one FITS file per map, e.g. ~/data/ocm_dark/wk06/wk06_1330_1340_900s_mean.fits
os.makedirs(OUT_DIR, exist_ok=True)
for name, m in [('mean', mean), ('median', median), ('std', std), ('std_mad', std_mad), ('std_excess', std_excess)]:
    out = f'{OUT_DIR}/{TEL}_{N_START}_{N_STOP}_{EXPTIME}s_{name}.fits'
    fits.writeto(out, m, hdr, overwrite=True)
    print(out)